In [ ]:
# Kaggriculture RL corpus -- DELTA extraction (generated by python/delta.py; do not hand-edit the pushed copy).
# Runs the Rust `corpus-extract` (dataset debmalya84/kaggriculture-rl-bin) in slim mode over the
# mounted inputs and writes /kaggle/working/{slim/, run.json}. Only engine 1.32.7 episodes are kept
# (the extractor's default --engine). slim/manifest.txt lists every output file so the downloader can
# fetch exactly what is missing; run.json carries finished_utc so a stale output is never filed.
import datetime, glob, json, os, shutil, subprocess, time
T0 = time.time()
KIND = 'gm'            # 'daily' | 'gm'
EXTRA = ['--since', '2026-09-20', '--until', '2026-09-28']          # e.g. ['--since', '2026-09-21', '--until', '2026-09-24'] for gm
PLAN = {'dates': ['2026-09-20', '2026-09-21', '2026-09-22', '2026-09-25', '2026-09-26', '2026-09-27', '2026-09-28'], 'since': '2026-09-20', 'until': '2026-09-28', 'gm_version': '2026-09-30 00:43:36.270000', 'coverage': {'2026-09-20': 1.0, '2026-09-21': 1.0, '2026-09-22': 1.0, '2026-09-25': 0.944, '2026-09-26': 0.952, '2026-09-27': 0.859, '2026-09-28': 0.513}}            # what the planner asked for (echoed into run.json)
THREADS = "4"
os.makedirs('/kaggle/working/bin', exist_ok=True)
src = sorted(glob.glob('/kaggle/input/**/corpus-extract', recursive=True))[0]
BIN = '/kaggle/working/bin/corpus-extract'
shutil.copy(src, BIN); os.chmod(BIN, 0o755)
sha = open(os.path.join(os.path.dirname(src), 'SHA256.txt')).read().strip()
print('binary', src, sha, flush=True)
if KIND == 'gm':
    gm = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/episodes.csv', recursive=True))
    assert gm, 'GM dataset not mounted'
    args = ['--gm', gm[0]]
else:
    days = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/manifest.csv', recursive=True)
                   if 'kaggriculture-episodes-20' in p})
    print('daily dirs:', len(days), days, flush=True)
    assert days, 'no daily datasets mounted'
    args = sum((['--daily', d] for d in days), [])
OUT = '/kaggle/working/slim'
cmd = [BIN, '--mode', 'slim', '--out', OUT, '--threads', THREADS] + args + EXTRA
print(' '.join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
files = sorted(os.path.relpath(f, '/kaggle/working') for f in glob.glob(OUT + '/**/*', recursive=True) if os.path.isfile(f))
with open(os.path.join(OUT, 'manifest.txt'), 'w') as fh:
    fh.write('\n'.join(files + ['slim/manifest.txt']) + '\n')
files.append('slim/manifest.txt')
tot = sum(os.path.getsize(os.path.join('/kaggle/working', f)) for f in files)
json.dump({'rc': rc, 'secs': round(time.time() - T0, 1), 'bytes': tot, 'files': len(files), 'kind': KIND,
           'extra': EXTRA, 'plan': PLAN, 'binary_sha': sha,
           'finished_utc': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')},
          open('/kaggle/working/run.json', 'w'), indent=1)
shutil.rmtree('/kaggle/working/bin', ignore_errors=True)
print('RC', rc, 'files', len(files), 'bytes', tot, 'secs', round(time.time() - T0, 1), flush=True)
assert rc == 0, 'corpus-extract failed'
